In [3]:
from collections import Counter
import xml.etree.ElementTree as ET
tree = ET.parse('Block_1.xml')
root = tree.getroot()

In [4]:
''' O usuario fornecera os dados da seguinte forma[
[(Nome_elemento,tipo_elemento,subtipo_elemento,0),aresta_anterior, aresta_posterior]
[(Contato1,contato,1,0),0,1]
A aresta anterior deverá ser zero SEMPRE que o elemento estiver diretamente ligado ao barramento energizado
]'''
def diferenciar_nomes(dados_ladder):
    '''Funçao de pre processamento para retirar nomes repetidos e ordenar o problema'''
    contador = {}
    for x in dados_ladder:
        elemento = x[0]
        if elemento in contador:
            contador[elemento] +=1
        else:
            contador[elemento] = 1
        repeticoes = contador[elemento] -1
        x[0] = x[0][:-1] + (1*repeticoes,)
    return(dados_ladder)

In [5]:
dados_ladder_otimizados = [
    [("cont1",'contato',"1",0), 0, 1],
    [("cont3",'contato',"1",0), 0, 2],
    [("cont4",'contato',"1",0), 0, 2],
    [("cont9",'contato',"1",0), 0, 4],
    [("cont9",'contato',"1",0), 1, 2],
    [("cont10",'contato',"1",0), 4, 3],
    [("cont6",'contato',"1",0), 3, 5],
    [("bobina1",'contato',"1",0), 5, 6],
    [("cont7",'contato',"1",0), 3, 7],
    [("cont8",'contato',"1",0), 7, 8],
    [("bobina2",'contato',"1",0), 8, 9],
    [("cont5",'contato',"1",0), 2, 3]  # <--- O elemento central jogado no final da lista
]


In [6]:
dados_ladder_otimizados = diferenciar_nomes(dados_ladder_otimizados)
dados_ladder_otimizados



[[('cont1', 'contato', '1', 0), 0, 1],
 [('cont3', 'contato', '1', 0), 0, 2],
 [('cont4', 'contato', '1', 0), 0, 2],
 [('cont9', 'contato', '1', 0), 0, 4],
 [('cont9', 'contato', '1', 1), 1, 2],
 [('cont10', 'contato', '1', 0), 4, 3],
 [('cont6', 'contato', '1', 0), 3, 5],
 [('bobina1', 'contato', '1', 0), 5, 6],
 [('cont7', 'contato', '1', 0), 3, 7],
 [('cont8', 'contato', '1', 0), 7, 8],
 [('bobina2', 'contato', '1', 0), 8, 9],
 [('cont5', 'contato', '1', 0), 2, 3]]

In [5]:
# ==========================================
# ÁREA DE TESTE
# ==========================================
# Formato Otimizado: [Elemento, Nó_Anterior, Nó_Posterior]
# Vou manter a lista bagunçada para provar que a matemática organiza sozinha!
dados_ladder_otimizados = [
    ["cont1", 0, 1],
    ["cont3", 0, 2],
    ["cont4", 0, 2],
    ["cont9_inf", 0, 4],
    ["cont9_sup", 1, 2],
    ["cont10", 4, 3],
    ["cont6", 3, 5],
    ["bobina1", 5, 6],
    ["cont7", 3, 7],
    ["cont8", 7, 8],
    ["bobina2", 8, 9],
    ["cont5", 2, 3]  # <--- O elemento central jogado no final da lista
]

# Executando a função
elementos_nomeados = gerar_chaves_ladder(dados_ladder_otimizados)

'''# Imprimindo o resultado formatado
print(f"{'CHAVE':<7} | {'ELEMENTO':<12} | {'LINHA VIRTUAL':<15} | {'CONEXÃO (In -> Out)'}")
print("-" * 65)
for item in elementos_nomeados:
    print(f" {item['Chave']:02d}     | {item['Elemento']:<12} | Linha {item['Linha_Virtual']:<10} | Nó {item['No_in']} -> Nó {item['No_out']}")'''

'# Imprimindo o resultado formatado\nprint(f"{\'CHAVE\':<7} | {\'ELEMENTO\':<12} | {\'LINHA VIRTUAL\':<15} | {\'CONEXÃO (In -> Out)\'}")\nprint("-" * 65)\nfor item in elementos_nomeados:\n    print(f" {item[\'Chave\']:02d}     | {item[\'Elemento\']:<12} | Linha {item[\'Linha_Virtual\']:<10} | Nó {item[\'No_in\']} -> Nó {item[\'No_out\']}")'

In [7]:
def gerar_sequencia_ladder_com_nos(lista_elementos):
    elementos_dict = {}
    
    # 1. REGISTRO [Nome, No_in, No_out]
    for i, el in enumerate(lista_elementos):
        elementos_dict[i] = {
            "nome": el[0], 
            "no_in": el[1], 
            "no_out": el[2],
            "predecessores": set(), 
            "sucessores": set(),
            "indice_original": i,
            "y_score": None 
        }

    # 2. MAPEAMENTO DOS FIOS
    saidas_por_no = {}
    for i, el in elementos_dict.items():
        if el["no_out"] not in saidas_por_no: 
            saidas_por_no[el["no_out"]] = []
        saidas_por_no[el["no_out"]].append(i)

    for i, el in elementos_dict.items():
        if el["no_in"] in saidas_por_no:
            for pred_id in saidas_por_no[el["no_in"]]:
                el["predecessores"].add(pred_id)
                elementos_dict[pred_id]["sucessores"].add(i)

    # 3. CÁLCULO DA LINHA VIRTUAL
    raizes = [i for i, el in elementos_dict.items() if len(el["predecessores"]) == 0]
    raizes.sort(key=lambda x: elementos_dict[x]["indice_original"])
    
    for linha, id_raiz in enumerate(raizes):
        elementos_dict[id_raiz]["y_score"] = linha

    grau_entrada_y = {i: len(el["predecessores"]) for i, el in elementos_dict.items()}
    fila_y = raizes[:]
    
    while fila_y:
        atual_id = fila_y.pop(0)
        atual = elementos_dict[atual_id]
        
        for suc_id in atual["sucessores"]:
            suc = elementos_dict[suc_id]
            if suc["y_score"] is None:
                suc["y_score"] = atual["y_score"]
            else:
                suc["y_score"] = min(suc["y_score"], atual["y_score"])
            
            grau_entrada_y[suc_id] -= 1
            if grau_entrada_y[suc_id] == 0:
                fila_y.append(suc_id)

    # 4. ORDENAÇÃO TOPOLÓGICA
    grau_entrada = {i: len(el["predecessores"]) for i, el in elementos_dict.items()}
    prontos = [i for i, el in elementos_dict.items() if grau_entrada[i] == 0]
    
    resultado = [] 
    
    while prontos:
        prontos.sort(key=lambda x: (elementos_dict[x]["y_score"], elementos_dict[x]["indice_original"]))
        
        atual_id = prontos.pop(0)
        atual = elementos_dict[atual_id]
        
        # AQUI ESTÁ A MUDANÇA: Guarda a sub-lista completa com nome e nós
        resultado.append([atual["nome"], atual["no_in"], atual["no_out"]])
        
        for suc_id in atual["sucessores"]:
            grau_entrada[suc_id] -= 1
            if grau_entrada[suc_id] == 0:
                prontos.append(suc_id)
                
    return resultado

# ==========================================
# TESTE DA NOVA SAÍDA
# ==========================================
'''dados_ladder_otimizados = [
    ["cont1", 0, 1],
    ["cont3", 0, 2],
    ["cont4", 0, 2],
    ["cont9_inf", 0, 4],
    ["cont9_sup", 1, 2],
    ["cont10", 4, 3],
    ["cont6", 3, 5],
    ["bobina1", 5, 6],
    ["cont7", 3, 7],
    ["cont8", 7, 8],
    ["bobina2", 8, 9],
    ["cont5", 2, 3]  
]

lista_ordenada = gerar_sequencia_ladder_com_nos(dados_ladder_otimizados)

# Imprimindo item por item para visualizar melhor a estrutura
print("Sua lista final para o gerador de XML:")
print("[\n  " + ",\n  ".join(str(item) for item in lista_ordenada) + "\n]")'''

'dados_ladder_otimizados = [\n    ["cont1", 0, 1],\n    ["cont3", 0, 2],\n    ["cont4", 0, 2],\n    ["cont9_inf", 0, 4],\n    ["cont9_sup", 1, 2],\n    ["cont10", 4, 3],\n    ["cont6", 3, 5],\n    ["bobina1", 5, 6],\n    ["cont7", 3, 7],\n    ["cont8", 7, 8],\n    ["bobina2", 8, 9],\n    ["cont5", 2, 3]  \n]\n\nlista_ordenada = gerar_sequencia_ladder_com_nos(dados_ladder_otimizados)\n\n# Imprimindo item por item para visualizar melhor a estrutura\nprint("Sua lista final para o gerador de XML:")\nprint("[\n  " + ",\n  ".join(str(item) for item in lista_ordenada) + "\n]")'

In [8]:
resultado = gerar_sequencia_ladder_com_nos(dados_ladder_otimizados)
resultado

[[('cont1', 'contato', '1', 0), 0, 1],
 [('cont9', 'contato', '1', 1), 1, 2],
 [('cont3', 'contato', '1', 0), 0, 2],
 [('cont4', 'contato', '1', 0), 0, 2],
 [('cont5', 'contato', '1', 0), 2, 3],
 [('cont9', 'contato', '1', 0), 0, 4],
 [('cont10', 'contato', '1', 0), 4, 3],
 [('cont6', 'contato', '1', 0), 3, 5],
 [('bobina1', 'contato', '1', 0), 5, 6],
 [('cont7', 'contato', '1', 0), 3, 7],
 [('cont8', 'contato', '1', 0), 7, 8],
 [('bobina2', 'contato', '1', 0), 8, 9]]

In [9]:
def ordenar_tipo_elemento(dados):
    #Com a lista ja ordenada, agora vou ordenar o elemento para colocalo na part do meu xml
    contagem = Counter([x[2] for x in dados])
    vistos = {}
    resultado = []

    for item in dados:
        valor = item[2]

        # conta quantas vezes já vi esse valor
        vistos[valor] = vistos.get(valor, 0) + 1

        resultado.append(item)

        # se for a ÚLTIMA ocorrência e tiver repetição
        if contagem[valor] > 1 and vistos[valor] == contagem[valor]:
            resultado.append(['card',valor, contagem[valor]])

    return resultado

def definir_uid_part(lista_ordenada_com_card,dados):
    '''
    O objetivo dessa funcao é adicionar uma lista com um ou dois elementos que serao os os UIds do nome do elemento e do tipo do elemento.
    Se for uma junção de cardinalidade, ele só recebera um elemento para ser colocado ao part   
    Retorna a lista ordenada com:
    |[(nome,tipo,subtipo,comtagem elememtos iguais),no_entrada,no_saida,[uid_nome,uid_contado]| se nao for card
    |[card,numero_elemento_repetido, quantidade_de_repeticoes,[uid_contato]]
    '''
    tamanho_lista =  len(lista_ordenada_com_card)
    tamanho_card = len(dados)
    uid_inicial = 21
    uid_part =  21 + tamanho_card
    for x in range(len(lista_ordenada_com_card)):
        if lista_ordenada_com_card[x][0] == 'card':
            lista_ordenada_com_card[x].append([])
            lista_ordenada_com_card[x][-1].append(uid_part)
            uid_part +=1  
        else:
            lista_ordenada_com_card[x].append([])
            lista_ordenada_com_card[x][-1].append(uid_inicial)
            lista_ordenada_com_card[x][-1].append(uid_part)
            uid_inicial +=1
            uid_part +=1
    uid_inicial = 21
    '''for x in range(len(dados)):
        dados[x].append([])
        dados[x][-1].append(uid_inicial)
        uid_inicial +=1'''

    return(lista_ordenada_com_card,dados,uid_part)



In [10]:
lo_com_card = ordenar_tipo_elemento(resultado)

In [11]:
pre_xml = definir_uid_part(lo_com_card,resultado)
pre_xml

([[('cont1', 'contato', '1', 0), 0, 1, [21, 33]],
  [('cont9', 'contato', '1', 1), 1, 2, [22, 34]],
  [('cont3', 'contato', '1', 0), 0, 2, [23, 35]],
  [('cont4', 'contato', '1', 0), 0, 2, [24, 36]],
  ['card', 2, 3, [37]],
  [('cont5', 'contato', '1', 0), 2, 3, [25, 38]],
  [('cont9', 'contato', '1', 0), 0, 4, [26, 39]],
  [('cont10', 'contato', '1', 0), 4, 3, [27, 40]],
  ['card', 3, 2, [41]],
  [('cont6', 'contato', '1', 0), 3, 5, [28, 42]],
  [('bobina1', 'contato', '1', 0), 5, 6, [29, 43]],
  [('cont7', 'contato', '1', 0), 3, 7, [30, 44]],
  [('cont8', 'contato', '1', 0), 7, 8, [31, 45]],
  [('bobina2', 'contato', '1', 0), 8, 9, [32, 46]]],
 [[('cont1', 'contato', '1', 0), 0, 1, [21, 33]],
  [('cont9', 'contato', '1', 1), 1, 2, [22, 34]],
  [('cont3', 'contato', '1', 0), 0, 2, [23, 35]],
  [('cont4', 'contato', '1', 0), 0, 2, [24, 36]],
  [('cont5', 'contato', '1', 0), 2, 3, [25, 38]],
  [('cont9', 'contato', '1', 0), 0, 4, [26, 39]],
  [('cont10', 'contato', '1', 0), 4, 3, [27, 4

In [12]:
lista_nomear = []
lista_de = []#de = definir elemento
dicionario_elementos = {"bobina":"coil","contato":"Contact"}
dicionario_bobinas = {"1":"coil","c":"coil","r":"RCoil","s":"SCoil"}
dicionario_contatos = {"1":"Contact","c":"contact"}
dicionario_elementos_completo = {"Contact":dicionario_contatos,"coil":dicionario_bobinas}

def parts_total_inicio():
    root = ET.Element("Parts")
    #xml_string = ET.tostring(root, encoding="utf-8")
    #lista_nomear.append(xml_string)



def nomear_elemento(uid,nome,tipo='1',subtipo='1',extra = 1):
    '''Cria o codigo xml para o nome do elemento'''
    access = ET.Element("Access", Scope ="GlobalVariable", UId = str(uid) )
    symbol = ET.SubElement(access, "Symbol")
    ET.SubElement(symbol, "Component", Name=str(nome))
    ET.indent(access, space="    ", level=0)
    xml_string = ET.tostring(access, encoding='unicode')
    print(xml_string)
    lista_nomear.append(xml_string)
    return('Resolvido')

def retornar_string_elemento(tipo,subtipo = '1'):
    string_elemento = dicionario_elementos_completo[dicionario_elementos[tipo]][subtipo]
    return(string_elemento)

def definir_elemento(uid,nome, tipo,subtipo = '1'):
    '''Cria o codigo xml para o tipo de elemento associado ao nome'''
    part = ET.Element("Part", Name = retornar_string_elemento(tipo,subtipo), UId = str(uid))
    xml_string_2 = ET.tostring(part, encoding='unicode')
    #lista definir
    lista_de.append(xml_string_2)
    print(xml_string_2)

def parte_esquerda(uid,numero_cardinalidade):
    '''
    Cria mais um #part referenciando ao objeto O de adição de elementos em um mesmo no
    Responsavel pela cardinalidade ao definir parts
    ATENCAO: PRECISO DEFINIR A  ENTRADA NUMERO_CARDINALIDADE QUE VEM DE UMA OUTRA LISTA QUE EU JA FIZ
    '''
    part = ET.Element("Part", Name="O", UId=str(uid))
    template_value = ET.SubElement(part, "TemplateValue", Name="Card", Type="Cardinality")
    template_value.text = str(numero_cardinalidade)
    ET.indent(part, space="    ", level=0)
    xml_string = ET.tostring(part, encoding='unicode')
    lista_de.append(xml_string)
    print(xml_string)
    #return('Fim')


def escrever_elementos_xml_part(lista_com_cardinalidade):
    parts_total_inicio()
    for x in range(len(lista_com_cardinalidade)):
        if lista_com_cardinalidade[x][0] != 'card':
            nomear_elemento(lista_com_cardinalidade[x][-1][0],lista_com_cardinalidade[x][0][0])
    for x in range(len(lista_com_cardinalidade)):
        if lista_com_cardinalidade[x][0] == 'card':
            parte_esquerda(lista_com_cardinalidade[x][-1][0],lista_com_cardinalidade[x][2])
        else:
            definir_elemento(lista_com_cardinalidade[x][-1][-1],lista_com_cardinalidade[x][0][0],lista_com_cardinalidade[x][0][1],lista_com_cardinalidade[x][-0][2])


In [13]:
def juntar_xml_parts():
    """Junta as strings das listas e abraça tudo com a tag <Parts>"""
    
    # 1. Abre a tag no início do blocão
    xml_completo = "<Parts>\n"
    
    # 2. Adiciona todos os blocos <Access> gerados
    for trecho in lista_nomear:
        xml_completo += trecho + "\n"
        
    # 3. Adiciona todos os blocos <Part> gerados
    for trecho in lista_de:
        xml_completo += trecho + "\n"
        
    # 4. Fecha a tag no final do blocão
    xml_completo += "</Parts>"
    
    return xml_completo

In [16]:
print(juntar_xml_parts())

<Parts>
<Access Scope="GlobalVariable" UId="21">
    <Symbol>
        <Component Name="cont1" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="22">
    <Symbol>
        <Component Name="cont9" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="23">
    <Symbol>
        <Component Name="cont3" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="24">
    <Symbol>
        <Component Name="cont4" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="25">
    <Symbol>
        <Component Name="cont5" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="26">
    <Symbol>
        <Component Name="cont9" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="27">
    <Symbol>
        <Component Name="cont10" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="28">
    <Symbol>
        <Component Name="cont6" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="29">
    <Symbol>
        <Component Name="bobina1"

In [14]:
escrever_elementos_xml_part(pre_xml[0])

<Access Scope="GlobalVariable" UId="21">
    <Symbol>
        <Component Name="cont1" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="22">
    <Symbol>
        <Component Name="cont9" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="23">
    <Symbol>
        <Component Name="cont3" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="24">
    <Symbol>
        <Component Name="cont4" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="25">
    <Symbol>
        <Component Name="cont5" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="26">
    <Symbol>
        <Component Name="cont9" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="27">
    <Symbol>
        <Component Name="cont10" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="28">
    <Symbol>
        <Component Name="cont6" />
    </Symbol>
</Access>
<Access Scope="GlobalVariable" UId="29">
    <Symbol>
        <Component Name="bobina1" />
    

In [17]:
pre_xml[-1]

47

In [8]:
x = [1,2,3]
len(x)

3

In [24]:
'''
Bloco de implementação dos Wires
Necessário passar pelo bloco de implementação Part usando o UId que continuou do bloco anterior
'''
lista_wires = []
def barra_energizada(uid,lista_com_cardinalidade):
    #Funcao ligada ao powerrail. No futuro vou deixala como opcional para ser ativada ou não
    lista_wires.append([])
    wire = ET.Element("Wire", UId = str(uid))
    ET.SubElement(wire,"Powerrail")
    uid = uid + 1
    for x in range(len(lista_com_cardinalidade)):
        if lista_com_cardinalidade[x][0] != 'card' and lista_com_cardinalidade[x][1] == 0:
            ET.SubElement(wire,"NameCon", UId= str(lista_com_cardinalidade[x][-1][-1]) , Name="in")
    ET.indent(wire, space="  ")
    xml_output_w1 = ET.tostring(wire, encoding="unicode")
    lista_wires.append(xml_output_w1)
    print(lista_wires)
    return(lista_wires,uid)




In [21]:
pre_xml[0]

[[('cont1', 'contato', '1', 0), 0, 1, [21, 33]],
 [('cont9', 'contato', '1', 1), 1, 2, [22, 34]],
 [('cont3', 'contato', '1', 0), 0, 2, [23, 35]],
 [('cont4', 'contato', '1', 0), 0, 2, [24, 36]],
 ['card', 2, 3, [37]],
 [('cont5', 'contato', '1', 0), 2, 3, [25, 38]],
 [('cont9', 'contato', '1', 0), 0, 4, [26, 39]],
 [('cont10', 'contato', '1', 0), 4, 3, [27, 40]],
 ['card', 3, 2, [41]],
 [('cont6', 'contato', '1', 0), 3, 5, [28, 42]],
 [('bobina1', 'contato', '1', 0), 5, 6, [29, 43]],
 [('cont7', 'contato', '1', 0), 3, 7, [30, 44]],
 [('cont8', 'contato', '1', 0), 7, 8, [31, 45]],
 [('bobina2', 'contato', '1', 0), 8, 9, [32, 46]]]

In [25]:
a = barra_energizada(47,pre_xml[0])

[[], '<Wire UId="47">\n  <Powerrail />\n  <NameCon UId="33" Name="in" />\n  <NameCon UId="35" Name="in" />\n  <NameCon UId="36" Name="in" />\n  <NameCon UId="39" Name="in" />\n</Wire>']


In [26]:
print(a[0])

[[], '<Wire UId="47">\n  <Powerrail />\n  <NameCon UId="33" Name="in" />\n  <NameCon UId="35" Name="in" />\n  <NameCon UId="36" Name="in" />\n  <NameCon UId="39" Name="in" />\n</Wire>']
